# Telecom Revenue & Customer Churn SQL + Power BI Analytics Pipeline

**Tools:** SQL (SQLite3 / Window Functions / CTEs / CASE Segmentation), Python (Pandas), Power BI (DAX Measures & KPI Dashboard)  
**Dataset:** IBM Telco Customer Churn (7,043 raw rows -> 7,032 cleaned active/churned customer records)

In [17]:
import sqlite3
import pandas as pd

In [18]:
#import dataset
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)


In [19]:
#df.to_csv("IBM_Telco_Customer_Churn.csv", index=False)
#from google.colab import files
#files.download("IBM_Telco_Customer_Churn.csv")

In [20]:
# 1.Clean Raw Data for SQL Warehouse Loading
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.dropna(subset=['TotalCharges'], inplace=True)
df['ChurnFlag'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Load into in-memory SQLite relational database
conn = sqlite3.connect(':memory:')
df.to_sql('telco_customers', conn, index=False, if_exists='replace')

print(f"Loaded {len(df)} cleaned customer records into SQL table 'telco_customers'.")

Loaded 7032 cleaned customer records into SQL table 'telco_customers'.


## Query 1: Executive KPI Summary & Monthly Revenue at Risk (CTE + Aggregations)

In [21]:
q1 = """
WITH base_metrics AS (
    SELECT
        COUNT(customerID) AS total_customers,
        SUM(ChurnFlag) AS churned_customers,
        ROUND(SUM(MonthlyCharges), 2) AS total_mrr,
        ROUND(SUM(CASE WHEN ChurnFlag = 1 THEN MonthlyCharges ELSE 0 END), 2) AS churned_mrr,
        ROUND(AVG(MonthlyCharges), 2) AS avg_arpu
    FROM telco_customers
)
SELECT
    total_customers,
    churned_customers,
    ROUND(100.0 * churned_customers / total_customers, 2) AS overall_churn_rate_pct,
    total_mrr,
    churned_mrr,
    ROUND(100.0 * churned_mrr / total_mrr, 2) AS revenue_attrition_pct,
    avg_arpu
FROM base_metrics;
"""
pd.read_sql_query(q1, conn)

,total_customers,churned_customers,overall_churn_rate_pct,total_mrr,churned_mrr,revenue_attrition_pct,avg_arpu
0,7032,1869,26.58,455661.0,139130.85,30.53,64.8


## Query 2: Contract Type & Internet Service Churn Segmentation (GROUP BY + Window Functions)

In [22]:
q2 = """
WITH segment_summary AS (
    SELECT
        Contract,
        InternetService,
        COUNT(customerID) AS customers,
        SUM(ChurnFlag) AS churned,
        ROUND(100.0 * SUM(ChurnFlag) / COUNT(customerID), 2) AS churn_rate_pct,
        ROUND(SUM(CASE WHEN ChurnFlag = 1 THEN MonthlyCharges ELSE 0 END), 2) AS monthly_revenue_lost
    FROM telco_customers
    GROUP BY Contract, InternetService
)
SELECT
    Contract,
    InternetService,
    customers,
    churned,
    churn_rate_pct,
    monthly_revenue_lost,
    RANK() OVER (ORDER BY monthly_revenue_lost DESC) AS revenue_loss_rank
FROM segment_summary
ORDER BY revenue_loss_rank;
"""
pd.read_sql_query(q2, conn)

,Contract,InternetService,customers,churned,churn_rate_pct,monthly_revenue_lost,revenue_loss_rank
0,Month-to-month,Fiber optic,2128,1162,54.61,100482.00,1
1,Month-to-month,DSL,1223,394,32.22,18367.25,2
2,One year,Fiber optic,539,104,19.29,10571.95,3
3,One year,DSL,570,53,9.30,3356.25,4
4,Two year,Fiber optic,429,31,7.23,3246.10,5
5,Month-to-month,No,524,99,18.89,1997.85,6
6,Two year,DSL,623,12,1.93,805.70,7
7,One year,No,363,9,2.48,190.25,8
8,Two year,No,633,5,0.79,113.50,9


## Query 3: Tenure Cohort Retention & High-Value At-Risk Customer Ranking (CASE + DENSE_RANK)

In [23]:
q3 = """
WITH tenure_cohorts AS (
    SELECT
        CASE
            WHEN tenure <= 12 THEN '0-12 Months (New)'
            WHEN tenure <= 24 THEN '13-24 Months'
            WHEN tenure <= 48 THEN '25-48 Months'
            ELSE '49+ Months (Loyal)'
        END AS tenure_group,
        COUNT(customerID) AS total_customers,
        SUM(ChurnFlag) AS churned_customers,
        ROUND(100.0 * SUM(ChurnFlag) / COUNT(customerID), 2) AS churn_rate_pct,
        ROUND(AVG(MonthlyCharges), 2) AS avg_monthly_bill
    FROM telco_customers
    GROUP BY tenure_group
)
SELECT * FROM tenure_cohorts ORDER BY churn_rate_pct DESC;
"""
pd.read_sql_query(q3, conn)

,tenure_group,total_customers,churned_customers,churn_rate_pct,avg_monthly_bill
0,0-12 Months (New),2175,1037,47.68,56.17
1,13-24 Months,1024,294,28.71,61.36
2,25-48 Months,1594,325,20.39,65.93
3,49+ Months (Loyal),2239,213,9.51,73.95


## Query 4: Top High-Revenue Active Customers at Risk (Window Partitioning for Retention Targeting)

In [24]:
q4 = """
WITH ranked_active AS (
    SELECT
        customerID,
        Contract,
        InternetService,
        tenure,
        MonthlyCharges,
        DENSE_RANK() OVER (PARTITION BY Contract ORDER BY MonthlyCharges DESC) AS bill_rank_in_contract
    FROM telco_customers
    WHERE ChurnFlag = 0 AND Contract = 'Month-to-month' AND tenure <= 12
)
SELECT * FROM ranked_active WHERE bill_rank_in_contract <= 5 LIMIT 10;
"""
pd.read_sql_query(q4, conn)

,customerID,Contract,InternetService,tenure,MonthlyCharges,bill_rank_in_contract
0,3292-PBZEJ,Month-to-month,Fiber optic,11,111.40,1
1,5760-IFJOZ,Month-to-month,Fiber optic,3,107.95,2
2,2081-VEYEH,Month-to-month,Fiber optic,3,107.95,2
3,6734-GMPVK,Month-to-month,Fiber optic,5,105.30,3
4,7145-FEJWU,Month-to-month,Fiber optic,12,105.30,3
5,2018-PZKMU,Month-to-month,Fiber optic,9,103.10,4
6,9547-ITEFG,Month-to-month,Fiber optic,9,102.60,5


In [25]:
df.to_csv("telco_churn_cleaned.csv", index=False)
from google.colab import files
files.download("telco_churn_cleaned.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>